*Unlike other scripts, this is not pulling vaccine data at all. 
Anyone can be linked from patient to L2 data!*

In [ ]:
import pandas as pd
import zipfile
import numpy as np
import pyreadr
import os

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm
import matplotlib.ticker as ticker
import matplotlib.lines as mlines

### Read in L2 voter data

In [ ]:
zf = zipfile.ZipFile("/share/pi/deho-pi/AFC/l2/l2_all_states.csv.zip") 

In [ ]:
import zipfile
import pandas as pd

zip_path = "/share/pi/deho-pi/AFC/l2/l2_all_states.csv.zip"

with zipfile.ZipFile(zip_path) as zf:
    with zf.open("l2_all_states.csv") as f:
        # Read only the column of interest
        df = pd.read_csv(
            f,
            usecols=["Voters_BirthDate"],
            parse_dates=["Voters_BirthDate"]
        )

latest_date = df["Voters_BirthDate"].max()
print(latest_date)


In [ ]:
sorted(df['Voters_BirthDate'])

In [ ]:
l2 = pd.read_csv(zf.open('l2_all_states.csv'))

Voters in L2 dataset

In [ ]:
# number of voters
len(l2)

### Read in and link dataset with DOB and Geography

In [ ]:
# use patient baseline from AFC (has ID to match with vaccination data, plus all covariates)
patient_baseline = pd.read_csv('/share/pi/deho-pi/AFC/BQ/GeneratedPatientBaseline_0724.csv.gz')
patient_baseline = patient_baseline[['patientuid', 'dob', 'fips_state', 'fips_county', 'state']]

# restrict to patients who have a date of birth
patient_baseline = patient_baseline[patient_baseline.dob.notnull()]

### Read in and merge on names

In [ ]:
patient_names = pyreadr.read_r("/share/pi/deho-pi/AFC/intermediateAFCData/exploration_DO/patient_baseline_bisg2.rds")
patient_names = patient_names[None]
patient_names = patient_names[['patientuid', 'first', 'last', 'state']]

# create merged covariates, vaccination status, name dataset
patient = patient_baseline.merge(patient_names, on='patientuid', how='left')

# require an existing first and last name
patient = patient[patient['first'].notnull()]
patient = patient[patient['last'].notnull()]
patient['state'] = patient['state_x']
patient = patient.drop(columns=['state_x', 'state_y'])

In [ ]:
len(patient)

In [ ]:
# count number patients vaccinated, unvaccinated
patient.columns

### Clean/Standardize firstname, lastname, DOB, state columns

In [ ]:
patient['first'] = patient['first'].str.upper()
patient['first']

In [ ]:
patient['last'] = patient['last'].str.upper()
patient['last']

In [ ]:
patient.dob

In [ ]:
patient['birthdate'] = pd.to_datetime(patient['dob']).dt.date
patient[['dob', 'birthdate']]

In [ ]:
l2.loc[:,'first'] = l2['Voters_FirstName'].str.upper()
l2.loc[:,'last'] = l2['Voters_LastName'].str.upper()
l2.loc[:,'state'] = l2['Residence_Addresses_State']
l2.loc[:,'first_init'] = l2['first'].str[0]
l2.loc[:,'dob'] = l2['Voters_BirthDate']

In [ ]:
l2 = l2.copy()
l2.drop(columns=['Residence_Addresses_State', 
                'Residence_Addresses_AddressLine', 
                'Voters_FirstName', 
                'Voters_LastName',
                'Voters_BirthDate'], inplace=True)

In [ ]:
l2['birthdate'] = pd.to_datetime(l2['dob']).dt.date

In [ ]:
# create birth month, birth year columns in the two datasets we're merging
l2['month'] = pd.to_datetime(l2['dob']).dt.month
l2['year'] = pd.to_datetime(l2['dob']).dt.year

In [ ]:
# create birth month, birth year columns in the two datasets we're merging
patient['month'] = pd.to_datetime(patient['dob']).dt.month
patient['year'] = pd.to_datetime(patient['dob']).dt.year

In [ ]:
l2['month'] = l2['month'].astype('Int64')
l2['year'] = l2['year'].astype('Int64')

In [ ]:
# keep_duplicates should be set to True if, later on, we might be able to uniquely match a duplicate from a previous run, 
# so if later on we have an additional layer of specificity (while also losing one or more layers of specificity)
def merge_and_analyze(df1, df2, merge_columns, keep_duplicates = False):
    # left merge based on the column names
    merged = pd.merge(df1, df2, how='left', on=merge_columns, indicator=True)

    # find unique matches, duplicates, and no matches
    unique_matches = merged[merged['_merge'] == 'both']
    duplicate_patients = unique_matches[unique_matches.duplicated(subset = ['patientuid'], keep=False)]
    duplicate_voters = unique_matches[unique_matches.duplicated(subset = ['Unnamed: 0'], keep=False)]
    no_matches = merged[merged['_merge'] == 'left_only']

    # remove duplicates from unique matches
    unique_matches = unique_matches.drop_duplicates(subset=['patientuid'], keep = False)
    unique_matches = unique_matches.drop_duplicates(subset=['Unnamed: 0'], keep = False)
    
    # Print number of unique, duplicate, and no matches
    print(f"Number of unique matches: {len(unique_matches)}")
    print(f"Number of duplicate patients: {len(duplicate_patients['patientuid'].unique())}")
    print(f"Number of duplicate voters: {len(duplicate_voters['Unnamed: 0'].unique())}")
    print(f"Number of no matches: {len(no_matches)}")
    
    # keep_duplicates = True means the cases leading to duplicates will be kept in the "no matches" returned datasets
    if keep_duplicates:
        # Remove rows that matched from unique only
        patientids_to_remove = unique_matches['patientuid'].unique()
        voterids_to_remove = unique_matches['Unnamed: 0'].unique()
    else:
        # we remove all patients that led to duplicate patients, and voters that led to duplicate voters
        # we don't remove voters that matched to the same patient, or patients that matched to the same voter.
        patientids_to_remove = pd.concat([unique_matches['patientuid'], 
                                          duplicate_patients['patientuid'], 
                                          duplicate_voters['patientuid']]).unique()
        voterids_to_remove = pd.concat([unique_matches['Unnamed: 0'], 
                                        duplicate_patients['Unnamed: 0'], 
                                       duplicate_voters['Unnamed: 0']]).unique()
        
    # drop columns from the "no matches" dataset -- either
    df1_no_matches = df1[~df1['patientuid'].isin(patientids_to_remove)]
    df2_no_matches = df2[~df2['Unnamed: 0'].isin(voterids_to_remove)]
    
    print(f"Number of patient ids removed: {len(patientids_to_remove)}")
    print(f"Number of voter ids removed: {len(voterids_to_remove)}")
    
    # checksum -- general
    sum_patients = len(unique_matches)+len(no_matches)+len(pd.concat([duplicate_patients['patientuid'],duplicate_voters['patientuid']]).unique())
    if (sum_patients != len(df1)):
        print("Number of unique patients output does not equal number of patients input: 1")
    
    # if we didn't keep duplicates in the output, then this should all add up
    # because unique matches has unique ones, no matches has all of the duplicates removed, 
    # and those duplicates are stored in duplicate patients and duplicate voters
    if not keep_duplicates:
        sum_patients2 = len(unique_matches)+len(df1_no_matches)+len(pd.concat([duplicate_patients['patientuid'],duplicate_voters['patientuid']]).unique())
        if (sum_patients2 != len(df1)):
            print("Number of unique patients output does not equal number of patients input: 2")
        
    # Return the results
    return unique_matches, duplicate_patients, duplicate_voters, df1_no_matches, df2_no_matches


In [ ]:
# Version of the above function that takes a random member from duplicates, cuts the rest of duplicates out
# keep_duplicates should be set to True if, later on, we might be able to uniquely match a duplicate from a previous run, 
# so if later on we have an additional layer of specificity (while also losing one or more layers of specificity)
def merge_and_analyze_random(df1, df2, merge_columns):
    # left merge based on the column names
    merged = pd.merge(df1, df2, how='left', on=merge_columns, indicator=True)

    # find unique matches, duplicates, and no matches
    unique_matches = merged[merged['_merge'] == 'both']
    no_matches = merged[merged['_merge'] == 'left_only']

    # remove all duplicates except one (chosen randomly)
    unique_matches = unique_matches.sample(frac=1).reset_index(drop=True)  # Shuffle the rows randomly
    unique_matches = unique_matches.drop_duplicates(subset=['patientuid'], keep = 'first')
    unique_matches = unique_matches.drop_duplicates(subset=['Unnamed: 0'], keep = 'first')
    
    # Print number of unique, duplicate, and no matches
    print(f"Number of unique matches: {len(unique_matches)}")
    print(f"Number of no matches: {len(no_matches)}")
    
    patientids_to_remove = unique_matches['patientuid']
    voterids_to_remove = unique_matches['Unnamed: 0']
        
    # drop columns from the "no matches" dataset -- either
    df1_no_matches = df1[~df1['patientuid'].isin(patientids_to_remove)]
    df2_no_matches = df2[~df2['Unnamed: 0'].isin(voterids_to_remove)]
    
    print(f"Number of patient ids removed: {len(patientids_to_remove)}")
    print(f"Number of voter ids removed: {len(voterids_to_remove)}")
       
    # Return the results
    return unique_matches, df1_no_matches, df2_no_matches


## Create Tier 1
Unique match on exact first name, last name, state, and birthdate (Tier 1)

In [ ]:
tier_1, tier_1_duplicate_patients, tier_1_duplicate_voters, \
patient_tier2_input, l2_tier2_input = merge_and_analyze(patient, 
                                                      l2, 
                                                      ['first', 'last', 'state', 'birthdate'])

## Create Tier 2
Not in Tier 1, Unique match on exact first name, last name, state, birth month, year

In [ ]:
tier_2, tier_2_duplicate_patients, tier_2_duplicate_voters, \
patient_tier3_input, l2_tier3_input = merge_and_analyze(patient_tier2_input,
                                                      l2_tier2_input, 
                                                      ['first', 'last', 'state', 'month', 'year'], 
                                                      keep_duplicates = True)

## Create Tier 3
Not in Tier 2, Unique match on exact first name, last name, state, and birth year

In [ ]:
tier_3, tier_3_duplicate_patients, tier_3_duplicate_voters, \
patient_tier4_input, l2_tier4_input = merge_and_analyze(patient_tier3_input, 
                                                      l2_tier3_input, 
                                                      ['first', 'last', 'state', 'year'], 
                                                       keep_duplicates = True)

## Create Tier 4
Not in Tier 3, Unique match on exact first name, last name, birth date

In [ ]:
tier_4, tier_4_duplicate_patients, tier_4_duplicate_voters, \
patient_tier5_input, l2_tier5_input  = merge_and_analyze(patient_tier4_input, 
                                                       l2_tier4_input, 
                                                       ['first', 'last', 'birthdate'], 
                                                       keep_duplicates = True)

## Create Tier 5
Not in Tier 4, Unique match on exact first name, last name, and birth month, year

In [ ]:
tier_5, tier_5_duplicate_patients, tier_5_duplicate_voters, \
patient_tier6_input, l2_tier6_input  = merge_and_analyze(patient_tier5_input, 
                                                       l2_tier5_input, 
                                                       ['first', 'last', 'month', 'year'])

## Create Tier 6
Not in Tier 5, Unique match on exact first name, last name, and birth year

In [ ]:
tier_6, tier_6_duplicate_patients, tier_6_duplicate_voters, \
patient_tier7_input, l2_tier7_input  = merge_and_analyze(patient_tier6_input,
                                                       l2_tier6_input, 
                                                       ['first', 'last', 'year'])

## Save this data
Concatenate all of the unique matches and put them in one dataset, which we save

In [ ]:
tier_1['tier'] = 1
tier_2['tier'] = 2
tier_3['tier'] = 3
tier_4['tier'] = 4
tier_5['tier'] = 5
tier_6['tier'] = 6

When states don't match, we take the state from the voting data (state_y). When birthdates don't match, we take the birthday from the patient data, since the voting data has a lot more of the errors where everyone is born on the first.

In [ ]:
tier_1_filtered = tier_1[['patientuid', 'Unnamed: 0',
                         'Parties_Description',
                         'first', 'last',
                         'birthdate', 
                         'state', 'County', 
                         'fips_state', 'fips_county',
                         'Residence_Addresses_CensusTract',
                         'Residence_Addresses_CensusBlockGroup', 
                         'Residence_Addresses_Zip', 'tier']]

tier_2_filtered = tier_2[['patientuid', 'Unnamed: 0',
                         'Parties_Description',
                         'first', 'last',
                         'birthdate_x', 
                         'state', 'County', 
                         'fips_state', 'fips_county',
                         'Residence_Addresses_CensusTract',
                         'Residence_Addresses_CensusBlockGroup', 
                         'Residence_Addresses_Zip', 'tier']]
tier_2_filtered = tier_2_filtered.rename(columns={'birthdate_x': 'birthdate'})

tier_3_filtered = tier_3[['patientuid', 'Unnamed: 0',
                         'Parties_Description',
                         'first', 'last',
                         'birthdate_x', 
                         'state', 'County', 
                         'fips_state', 'fips_county',
                         'Residence_Addresses_CensusTract',
                         'Residence_Addresses_CensusBlockGroup', 
                         'Residence_Addresses_Zip', 'tier']]
tier_3_filtered = tier_3_filtered.rename(columns={'birthdate_x': 'birthdate'})

tier_4_filtered = tier_4[['patientuid', 'Unnamed: 0',
                         'Parties_Description',
                         'first', 'last',
                         'birthdate', 
                         'state_y', 'County', 
                         'fips_state', 'fips_county',
                         'Residence_Addresses_CensusTract',
                         'Residence_Addresses_CensusBlockGroup', 
                         'Residence_Addresses_Zip', 'tier']]
tier_4_filtered = tier_4_filtered.rename(columns={'state_y': 'state'})

tier_5_filtered = tier_5[['patientuid', 'Unnamed: 0',
                         'Parties_Description',
                         'first', 'last',
                         'birthdate_x', 
                         'state_y', 'County', 
                         'fips_state', 'fips_county',
                         'Residence_Addresses_CensusTract',
                         'Residence_Addresses_CensusBlockGroup', 
                         'Residence_Addresses_Zip', 'tier']]
tier_5_filtered = tier_5_filtered.rename(columns={'state_y': 'state', 
                                                  'birthdate_x' : 'birthdate'})

tier_6_filtered = tier_6[['patientuid', 'Unnamed: 0',
                         'Parties_Description',
                         'first', 'last',
                         'birthdate_x', 
                         'state_y', 'County', 
                         'fips_state', 'fips_county',
                         'Residence_Addresses_CensusTract',
                         'Residence_Addresses_CensusBlockGroup', 
                         'Residence_Addresses_Zip', 'tier']]
tier_6_filtered = tier_6_filtered.rename(columns={'state_y': 'state', 
                                                  'birthdate_x' : 'birthdate'})


In [ ]:
concat_tiers = pd.concat([tier_1_filtered, 
                          tier_2_filtered, 
                          tier_3_filtered, 
                          tier_4_filtered, 
                          tier_5_filtered, 
                          tier_6_filtered])

In [ ]:
# write to CSV
csv_filename = '/share/pi/deho-pi/AFC/l2_afc_match_032125.csv'
concat_tiers.to_csv(csv_filename, index=False)

In [ ]:
# zip CSV
zip_filename = '/share/pi/deho-pi/AFC/l2_afc_match_032125.csv.zip'

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(csv_filename, os.path.basename(csv_filename))

# remove large csv
os.remove(csv_filename)